In [17]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import json
import ipywidgets as widgets
from ipywidgets import interact


In [18]:

# Configuração estética para relatórios técnicos
pio.renderers.default = "vscode"
layout_padrao = dict(template="simple_white", font=dict(family="Arial", size=12))

In [19]:
# --- CARGA DE PROPRIEDADES CRÍTICAS ---
try:
    with open('fluidos.json', 'r') as f:
        fluidos = json.load(f)
    print(f"✅ Banco de dados carregado: {len(fluidos)} componentes disponíveis.")
except FileNotFoundError:
    fluidos = {}
    print("⚠️ Erro: 'fluidos.json' não encontrado. Certifique-se de que o arquivo está no diretório raiz.")
except json.JSONDecodeError:
    print("⚠️ Erro: O arquivo 'fluidos.json' contém erros de formatação.")

✅ Banco de dados carregado: 13 componentes disponíveis.


In [20]:
def calcular_pvt(fluido_nome, T_c):
    if not fluidos: return print("Base de dados indisponível.")
    
    # Extração de propriedades
    p = fluidos[fluido_nome]
    T = T_c + 273.15  # Conversão para Kelvin
    R = 8.314
    
    # Vetor de pressão (MPa) - Aumentamos a resolução para 150 pontos
    pressões_mpa = np.linspace(0.1, 50, 200)
    z_lista = []
    
    # Parâmetros constantes para a temperatura T
    Tr = T / p["Tc"]
    kappa = 0.37464 + 1.54226*p["omega"] - 0.26992*p["omega"]**2
    alpha = (1 + kappa * (1 - np.sqrt(Tr)))**2
    
    # Parâmetros a e b da EoS
    a_const = (0.45724 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha
    b_const = 0.07780 * R * p["Tc"] / p["Pc"]
    
    for P_mpa in pressões_mpa:
        P = P_mpa * 1e6 # Conversão para Pa
        
        # Parâmetros adimensionais A e B
        A = (a_const * P) / (R**2 * T**2)
        B = (b_const * P) / (R * T)
        
        # Polinômio Cúbico de Peng-Robinson
        # Z^3 + (B-1)Z^2 + (A-2B-3B^2)Z + (B^3+B^2-AB) = 0
        coefs = [1, (B - 1), (A - 2*B - 3*B**2), (B**3 + B**2 - A*B)]
        raizes = np.roots(coefs)
        
        # Filtro de estabilidade física:
        # 1. Raiz deve ser real
        # 2. Raiz deve ser maior que B (volume do gás > volume das moléculas)
        z_reais = raizes[np.isreal(raizes)].real
        z_validos = z_reais[z_reais > B]
        
        if len(z_validos) > 0:
            z_lista.append(np.max(z_validos)) # Seleciona fase vapor (predominante em T > Tc)
        else:
            z_lista.append(np.nan)

    # --- VISUALIZAÇÃO ---
    fig = go.Figure()
    
    # Curva de Peng-Robinson
    fig.add_trace(go.Scatter(
        x=pressões_mpa, y=z_lista, 
        name='Modelo Peng-Robinson',
        line=dict(color='#002D62', width=3)
    ))
    
    # Referência de Gás Ideal
    fig.add_hline(y=1.0, line_dash="dash", line_color="#D62728", 
                  annotation_text="Comportamento Ideal (Z=1)")
    
    fig.update_layout(
        title=f"<b>Fator de Compressibilidade (Z): {fluido_nome.upper()}</b><br>Isoterma: {T_c}°C",
        xaxis_title="Pressão (MPa)",
        yaxis_title="Fator Z",
        **layout_padrao
    )
    
    fig.show()

In [21]:

# Inicialização do Simulador
print("🧪 Simulador Termodinâmico PVT")

interact(
    calcular_pvt, 
    fluido_nome=widgets.Dropdown(
        options=sorted(list(fluidos.keys())), 
        description='Componente:',
        style={'description_width': 'initial'}
    ),
    T_c=widgets.IntSlider(
        min=-50, max=400, step=5, value=25, 
        description='Temperatura (°C):',
        style={'description_width': 'initial'},
        continuous_update=False
    )
)

🧪 Simulador Termodinâmico PVT


interactive(children=(Dropdown(description='Componente:', options=('CO2', 'benzeno', 'etano', 'i-butano', 'met…

<function __main__.calcular_pvt(fluido_nome, T_c)>